# NboDriver Showcase Notebook

This notebook demonstrates the current `NboDriver` API in VeloxChem. It is organized as a compact tour of the implemented layers: NAO/NPA diagnostics, MO-in-NAO analysis, NBO candidates, Lewis assignments, resonance alternatives, NRA/NRT density-fit weights, user priors, donor-acceptor diagnostics, and open-shell spin-resolved fitting.

The examples use small molecules and modest basis sets so the notebook remains practical to run during development. Larger basis sets are used only where they make the feature visible, such as Rydberg/acceptor diagnostics.

## 1. Imports and Helper Functions

The helper functions below keep the examples focused on the NBO API rather than repeated SCF boilerplate. Restricted Hartree-Fock is used for singlets, and unrestricted Hartree-Fock is used for radicals.

In [1]:
from collections import Counter
from pathlib import Path
import importlib.util
import sys

import numpy as np
import veloxchem as vlx

try:
    from veloxchem.scfunrestdriver import ScfUnrestrictedDriver
except ImportError:
    ScfUnrestrictedDriver = vlx.ScfUnrestrictedDriver

project_root = Path.cwd().resolve()
if project_root.name == 'docs':
    project_root = project_root.parent
nbodriver_path = project_root / 'src' / 'pymodule' / 'nbodriver.py'
if nbodriver_path.exists():
    spec = importlib.util.spec_from_file_location('veloxchem.nbodriver', nbodriver_path)
    nbodriver_module = importlib.util.module_from_spec(spec)
    sys.modules['veloxchem.nbodriver'] = nbodriver_module
    spec.loader.exec_module(nbodriver_module)
    vlx.NboDriver = nbodriver_module.NboDriver

np.set_printoptions(precision=6, suppress=True)
print('VeloxChem version:', vlx.__version__)

VeloxChem version: 1.0rc4


In [2]:
MOLECULES = {
    'water': '''
O   0.000000   0.000000   0.000000
H   0.758602   0.000000   0.504284
H  -0.758602   0.000000   0.504284
''',
    'methane': '''
C   0.000000   0.000000   0.000000
H   0.629118   0.629118   0.629118
H  -0.629118  -0.629118   0.629118
H  -0.629118   0.629118  -0.629118
H   0.629118  -0.629118  -0.629118
''',
    'ethylene': '''
C  -0.669500   0.000000   0.000000
C   0.669500   0.000000   0.000000
H  -1.232100   0.923000   0.000000
H  -1.232100  -0.923000   0.000000
H   1.232100   0.923000   0.000000
H   1.232100  -0.923000   0.000000
''',
    'formaldehyde': '''
C   0.000000   0.000000   0.000000
O   0.000000   0.000000   1.216723
H   0.000000   0.926436  -0.595599
H   0.000000  -0.926436  -0.595599
''',
    'co': '''
C   0.000000   0.000000   0.000000
O   0.000000   0.000000   1.128000
''',
    'no': '''
N   0.000000   0.000000   0.000000
O   0.000000   0.000000   1.150000
''',
    'allyl': '''
C  -1.280000   0.000000   0.000000
C   0.000000   0.000000   0.000000
C   1.280000   0.000000   0.000000
H  -1.880000   0.920000   0.000000
H  -1.880000  -0.920000   0.000000
H   0.000000   1.080000   0.000000
H   1.880000   0.920000   0.000000
H   1.880000  -0.920000   0.000000
''',
    'benzene': '''
C   1.396792   0.000000   0.000000
C   0.698396   1.209951   0.000000
C  -0.698396   1.209951   0.000000
C  -1.396792   0.000000   0.000000
C  -0.698396  -1.209951   0.000000
C   0.698396  -1.209951   0.000000
H   2.490290   0.000000   0.000000
H   1.245145   2.156660   0.000000
H  -1.245145   2.156660   0.000000
H  -2.490290   0.000000   0.000000
H  -1.245145  -2.156660   0.000000
H   1.245145  -2.156660   0.000000
''',
    'formate': '''
C   0.000000   0.000000   0.000000
O   1.250000   0.000000   0.000000
O  -1.250000   0.000000   0.000000
H   0.000000   1.100000   0.000000
''',
    'nitrate': '''
N   0.000000   0.000000   0.000000
O   1.240000   0.000000   0.000000
O  -0.620000   1.073872   0.000000
O  -0.620000  -1.073872   0.000000
''',
    'ozone': '''
O  -1.080000   0.000000   0.000000
O   0.000000   0.000000   0.000000
O   1.080000   0.000000   0.000000
'''
}

def build_molecule(name, charge=0, multiplicity=1):
    mol = vlx.Molecule.read_str(MOLECULES[name])
    mol.set_charge(charge)
    mol.set_multiplicity(multiplicity)
    return mol

def run_scf(molecule, basis_label='sto-3g'):
    basis = vlx.MolecularBasis.read(molecule, basis_label, ostream=None)
    if molecule.get_multiplicity() == 1:
        scf = vlx.ScfRestrictedDriver()
    else:
        scf = ScfUnrestrictedDriver()
    scf.ostream.mute()
    scf.xcfun = 'hf'
    scf.compute(molecule, basis)
    return basis, scf

def run_nbo(name, charge=0, multiplicity=1, basis_label='sto-3g', options=None, constraints=None):
    molecule = build_molecule(name, charge=charge, multiplicity=multiplicity)
    basis, scf = run_scf(molecule, basis_label=basis_label)
    nbo = vlx.NboDriver()
    nbo.verbose = False
    nbo.ostream.mute()
    results = nbo.compute(molecule, basis, scf.mol_orbs, options=options, constraints=constraints)
    return molecule, basis, scf, nbo, results

In [3]:
def candidate_counts(results, key='nbo_candidates'):
    return Counter(item.get('type', '?') for item in results.get(key, []))

def primary_counts(results):
    return results['primary']['counts']

def print_foundation_diagnostics(results):
    diag = results['diagnostics']
    print('Foundation invariants passed:', diag['foundation_invariants_passed'])
    print('Electron count error      :', f"{diag['electron_count_error']:.3e}")
    print('Charge conservation error :', f"{diag['charge_conservation_error']:.3e}")
    print('Orthonormality error      :', f"{diag['orthonormality_error']:.3e}")
    print('Density formula error     :', f"{diag['density_formula_error']:.3e}")

def print_assignment_summary(results):
    primary = results['primary']
    print('Primary counts        :', primary['counts'])
    print('Candidate counts      :', dict(candidate_counts(results)))
    print('Selected electrons    :', primary['selected_lewis_electron_count'])
    print('Target electrons      :', primary['target_electron_count'])
    print('Formal charges        :', np.array(primary['formal_charges']))
    print('Sigma/pi pairs        :', primary['sigma_electron_pairs'], '/', primary['pi_electron_pairs'])

def print_alternatives(results, limit=8):
    print(f"{'rank':>4} {'score_wt':>10} {'nra_wt':>10} {'score':>11} {'e_count':>8} {'pi bonds':<20} active")
    nra_by_rank = {}
    if 'nra' in results:
        nra_by_rank = {item['rank']: item for item in results['nra'].get('structures', [])}
    for alt in results.get('alternatives', [])[:limit]:
        nra_weight = nra_by_rank.get(alt['rank'], {}).get('nra_weight')
        nra_text = 'n/a' if nra_weight is None else f'{100*nra_weight:9.3f}%'
        pi_text = ', '.join(f'{i}-{j}' for i, j in alt.get('pi_bonds', [])) or 'none'
        active = Counter((c['type'], c.get('subtype', ''), c.get('electron_count', 2.0)) for c in alt.get('active_nbo_list', []))
        print(f"{alt['rank']:4d} {100*alt['weight']:9.3f}% {nra_text:>10} {alt['score']:11.5f} {alt['selected_lewis_electron_count']:8.1f} {pi_text:<20} {dict(active)}")

def print_nra_summary(results):
    nra = results.get('nra')
    if not nra:
        print('No NRA/NRT results in this calculation.')
        return
    print('NRA subspace       :', nra['subspace'])
    print('Fit metric         :', nra['fit_metric'])
    print('Spin fit           :', nra.get('spin_fit', 'none'))
    print('Weight sum         :', f"{sum(nra['weights']):.12f}")
    print('Residual           :', nra['residual_norm'])
    print('Total residual     :', nra.get('total_residual_norm'))
    print('Spin residual      :', nra.get('spin_residual_norm'))
    print('Prior              :', nra.get('prior'))
    for warning in nra.get('warnings', []):
        print('Warning            :', warning)

## 2. Water: NAO/NPA, NBO Assignment, and Donor-Acceptor Diagnostics

Water is a compact example for checking the NAO/NPA foundation, lone pairs, bond NBOs, Rydberg acceptors, and donor-acceptor diagnostics. `def2-svp` is used here because the acceptor space is more informative than in a minimal basis.

In [4]:
water = run_nbo('water', basis_label='def2-svp')
mol_water, basis_water, scf_water, nbo_water, res_water = water
print_foundation_diagnostics(res_water)
print()
print_assignment_summary(res_water)
print()
print(nbo_water.npa_report(level='summary', return_text=True))
print(nbo_water.nbo_report(level='summary', return_text=True))

Foundation invariants passed: True
Electron count error      : 2.487e-14
Charge conservation error : 2.556e-14
Orthonormality error      : 2.192e-14
Density formula error     : 0.000e+00

Primary counts        : {'CR': 1, 'LP': 2, 'BD': 2}
Candidate counts      : {'CR': 1, 'LP': 2, 'RY': 9, 'BD': 2, 'BD*': 2}
Selected electrons    : 10.0
Target electrons      : 10.0
Formal charges        : [0. 0. 0.]
Sigma/pi pairs        : 5 / 0

A Summary of Natural Population Analysis:

  Atom #  Natural Charge        Core     Valence     Rydberg       Total
--------------------------------------------------------------------------
 O  1        -0.10016     1.99842     6.10174     0.00000     8.10016
 H  2        +0.05008     0.00000     0.94958     0.00034     0.94992
 H  3        +0.05008     0.00000     0.94958     0.00034     0.94992
 * Total *        -0.00000     1.99842     8.00090     0.00067    10.00000
Natural Bond Orbital (NBO) Primary Summary
Lewis assignment from generated NBO candidates

In [5]:
da = res_water['donor_acceptor_diagnostics']
print('Diagnostic model:', da['model'])
print('Donors / acceptors / interactions:', da['donor_count'], da['acceptor_count'], da['interaction_count'])
print()
print(f"{'donor':>7} {'acceptor':>9} {'types':<12} {'coupling':>12} {'overlap':>12}")
for item in da['interactions'][:10]:
    types = f"{item['donor_type']}->{item['acceptor_type']}"
    print(f"{item['donor_index']:7d} {item['acceptor_index']:9d} {types:<12} {item['density_coupling']:12.6f} {item['overlap']:12.6f}")

Diagnostic model: nao_density_coupling
Donors / acceptors / interactions: 5 11 55

  donor  acceptor types            coupling      overlap
      1        16 CR->BD*          0.000000     0.000000
      1        14 CR->BD*         -0.000000     0.000000
      3        16 LP->BD*         -0.000000     0.000000
      2        10 LP->RY           0.000000     0.000000
     15        10 BD->RY          -0.000000     0.000000
     15         5 BD->RY          -0.000000     0.000000
      3        11 LP->RY           0.000000     0.000000
      2        16 LP->BD*          0.000000     0.000000
      3         4 LP->RY          -0.000000     0.000000
      2         4 LP->RY          -0.000000     0.000000


## 3. Simple Closed-Shell Molecules: Sigma, Pi, Core, and Lone-Pair Patterns

These examples are intentionally run with `sto-3g`: the goal is to show robust Lewis accounting and candidate counts quickly, not to inspect acceptor complements in detail.

In [6]:
simple_cases = ['methane', 'ethylene', 'formaldehyde', 'co']
simple_results = {}
print(f"{'case':<14} {'basis':<8} {'primary counts':<28} {'candidate counts'}")
for name in simple_cases:
    mol, basis, scf, nbo, results = run_nbo(name, basis_label='sto-3g')
    simple_results[name] = (mol, basis, scf, nbo, results)
    print(f"{name:<14} {'sto-3g':<8} {str(primary_counts(results)):<28} {dict(candidate_counts(results))}")

case           basis    primary counts               candidate counts
methane        sto-3g   {'CR': 1, 'BD': 4}           {'CR': 1, 'BD': 4, 'BD*': 4}
ethylene       sto-3g   {'CR': 2, 'BD': 6}           {'CR': 2, 'BD': 6, 'BD*': 6}
formaldehyde   sto-3g   {'CR': 2, 'LP': 2, 'BD': 4}  {'CR': 2, 'LP': 2, 'BD': 4, 'BD*': 4}
co             sto-3g   {'CR': 2, 'LP': 2, 'BD': 3}  {'CR': 2, 'LP': 2, 'BD': 3, 'BD*': 3}


In [7]:
mol_eth, basis_eth, scf_eth, nbo_eth, res_eth = simple_results['ethylene']
print_assignment_summary(res_eth)
print()
print('Ethylene selected pi bonds:', [pair for alt in res_eth['alternatives'][:1] for pair in alt.get('pi_bonds', [])])
print('Antibonding complements in candidate pool:', candidate_counts(res_eth)['BD*'])
print(nbo_eth.nbo_report(level='full', return_text=True))

Primary counts        : {'CR': 2, 'BD': 6}
Candidate counts      : {'CR': 2, 'BD': 6, 'BD*': 6}
Selected electrons    : 16.0
Target electrons      : 16.0
Formal charges        : [0. 0. 0. 0. 0. 0.]
Sigma/pi pairs        : 7 / 1

Ethylene selected pi bonds: [(1, 2)]
Antibonding complements in candidate pool: 6
Natural Bond Orbital (NBO) Primary Summary
Lewis assignment from generated NBO candidates.
Available layers: NAO/NPA, MO-in-NAO analysis, candidates, alternatives, diagnostics, and NRA/NRT.
Report order: CR, BD(sigma), BD(pi), LP, RY, then antibonding/other candidates.

Primary counts: BD=6, CR=2
Candidate pool: BD=6, BD*=6, CR=2
Selected pairs: 8/8  occupation sum=15.97504  score=15.97504
Primary alternative rank=1  Score weight=1.00000

 NBO Type       Atoms                 Occ  Polarization             Source
--------------------------------------------------------------------------------------------
   1 CR(core)   C1                1.99994  C1:100.0%                one-center

## 4. Allyl Cation, Radical, and Anion

The allyl family shows how the same resonance machinery handles closed-shell cation/anion cases and an open-shell radical. The radical example exposes one-electron `SOMO`, radical lone-pair, and spin-active `BD(pi)` candidates.

In [8]:
allyl_options = {
    'max_alternatives': 24,
    'pi_min_occupation': 0.05,
    'conjugated_pi_max_path': 2,
    'include_nra': True,
    'nra_subspace': 'pi',
    'nra_fit_metric': 'frobenius',
}
allyl_cases = {
    'allyl cation': {'charge': 1, 'multiplicity': 1},
    'allyl radical': {'charge': 0, 'multiplicity': 2},
    'allyl anion': {'charge': -1, 'multiplicity': 1},
}
allyl_results = {}
for label, spec in allyl_cases.items():
    allyl_results[label] = run_nbo('allyl', charge=spec['charge'], multiplicity=spec['multiplicity'], basis_label='sto-3g', options=allyl_options)
    _, _, _, _, results = allyl_results[label]
    print(label)
    print('  unpaired electrons:', results['unpaired_electrons'])
    print('  primary counts    :', results['primary']['counts'])
    print('  selected electrons:', results['primary']['selected_lewis_electron_count'])
    print('  candidate counts  :', dict(candidate_counts(results)))
    print('  NRA spin fit      :', results['nra']['spin_fit'])
    print()

allyl cation
  unpaired electrons: 0.0
  primary counts    : {'CR': 3, 'BD': 8}
  selected electrons: 22.0
  candidate counts  : {'CR': 3, 'RY': 2, 'BD': 12, 'BD*': 12}
  NRA spin fit      : none

allyl radical
  unpaired electrons: 0.9999999999999971
  primary counts    : {'CR': 3, 'SOMO': 1, 'BD': 8}
  selected electrons: 23.0
  candidate counts  : {'CR': 3, 'SOMO': 2, 'LP': 2, 'BD': 13, 'BD*': 12}
  NRA spin fit      : total_spin

allyl anion
  unpaired electrons: 0.0
  primary counts    : {'CR': 3, 'LP': 2, 'BD': 7}
  selected electrons: 24.0
  candidate counts  : {'CR': 3, 'LP': 2, 'BD': 10, 'BD*': 10}
  NRA spin fit      : none



In [9]:
_, _, _, nbo_radical, res_radical = allyl_results['allyl radical']
one_electron = [c for c in res_radical['nbo_candidates'] if c.get('electron_count', 2.0) == 1.0]
print(f"{'idx':>4} {'type':<6} {'subtype':<20} {'atoms':<10} {'occ':>9} {'spin':>9}")
for candidate in one_electron:
    print(f"{candidate['index']:4d} {candidate['type']:<6} {candidate.get('subtype',''):<20} {str(tuple(a+1 for a in candidate['atoms'])):<10} {candidate['occupation']:9.5f} {candidate.get('spin_occupation', 0.0):9.5f}")
print()
print_nra_summary(res_radical)
print()
print_alternatives(res_radical, limit=10)

 idx type   subtype              atoms            occ      spin
   2 SOMO   one-electron         (1,)         0.99286   0.70620
   3 LP     radical-lone-pair    (1,)         0.99286   0.70620
   6 SOMO   one-electron         (3,)         0.99286   0.70620
   7 LP     radical-lone-pair    (3,)         0.99286   0.70620
  32 BD     pi                   (1, 3)       1.00000   1.00000

NRA subspace       : pi
Fit metric         : frobenius
Spin fit           : total_spin
Weight sum         : 1.000000000000
Residual           : 0.7980179696653615
Total residual     : 0.4376526600209842
Spin residual      : 0.6673026517895629
Prior              : {'active': False, 'mode': 'regularized', 'strength': 0.0, 'input_weights': None, 'normalized_weights': None, 'fixed': False, 'warnings': []}

rank   score_wt     nra_wt       score  e_count pi bonds             active
   1    24.992%    12.500%    22.53967     23.0 2-3                  {('LP', 'radical-lone-pair', 1.0): 1, ('BD', 'pi', 2.0): 1}
   2

## 5. User-Specified Allyl Structures

The driver does not contain molecule-specific resonance templates. User input is supplied as ordinary constraints and priors. This allyl example defines three user-requested calculations: a left-localized pi structure, a right-localized pi structure, and the unconstrained two-structure active space.

In [10]:
user_allyl_structures = [
    {
        'name': 'left pi structure',
        'constraints': {'required_pi_bonds': [(1, 2)], 'allowed_pi_bonds': [(1, 2), (2, 3)]},
        'prior': {'pi:1-2': 1.0, 'pi:2-3': 0.0},
    },
    {
        'name': 'right pi structure',
        'constraints': {'required_pi_bonds': [(2, 3)], 'allowed_pi_bonds': [(1, 2), (2, 3)]},
        'prior': {'pi:1-2': 0.0, 'pi:2-3': 1.0},
    },
    {
        'name': 'two-structure active space',
        'constraints': {'allowed_pi_bonds': [(1, 2), (2, 3)]},
        'prior': {'pi:1-2': 0.5, 'pi:2-3': 0.5},
    },
]

allyl_user_options = dict(allyl_options)
allyl_user_options.update({
    'nra_prior_strength': 10.0,
})

allyl_user_results = {}
for spec in user_allyl_structures:
    options = dict(allyl_user_options)
    options['nra_prior_weights'] = spec['prior']
    allyl_user_results[spec['name']] = run_nbo(
        'allyl',
        charge=1,
        multiplicity=1,
        basis_label='sto-3g',
        constraints=spec['constraints'],
        options=options,
    )
    _, _, _, _, results = allyl_user_results[spec['name']]
    print(spec['name'])
    print('  user constraints:', spec['constraints'])
    print('  user prior      :', spec['prior'])
    print_alternatives(results, limit=6)
    print()

left pi structure
  user constraints: {'required_pi_bonds': [(1, 2)], 'allowed_pi_bonds': [(1, 2), (2, 3)]}
  user prior      : {'pi:1-2': 1.0, 'pi:2-3': 0.0}
rank   score_wt     nra_wt       score  e_count pi bonds             active
   1   100.000%   100.000%    20.81441     22.0 1-2, 1-2             {('BD', 'pi', 2.0): 2}

right pi structure
  user constraints: {'required_pi_bonds': [(2, 3)], 'allowed_pi_bonds': [(1, 2), (2, 3)]}
  user prior      : {'pi:1-2': 0.0, 'pi:2-3': 1.0}
rank   score_wt     nra_wt       score  e_count pi bonds             active
   1   100.000%   100.000%    20.81441     22.0 2-3, 2-3             {('BD', 'pi', 2.0): 2}

two-structure active space
  user constraints: {'allowed_pi_bonds': [(1, 2), (2, 3)]}
  user prior      : {'pi:1-2': 0.5, 'pi:2-3': 0.5}
rank   score_wt     nra_wt       score  e_count pi bonds             active
   1    50.000%    50.000%    21.51189     22.0 1-2                  {('BD', 'pi', 2.0): 1}
   2    50.000%    50.000%    21.51189

## 6. Benzene: User-Specified Pi-Bond Alternatives, NRA/NRT Weights, and Priors

Benzene is used here only as a compact resonance test case. The important point is general: the user supplies an `allowed_pi_bonds` list, and the driver enumerates electron-counted pi-bond alternatives from that list. The resulting table compares score/ranking weights with NRA/NRT density-fit weights.

In [11]:
ring_pi_bonds = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 1)]
cross_ring_pi_bonds = [(1, 4), (2, 5), (3, 6)]
benzene_constraints = {'allowed_pi_bonds': ring_pi_bonds + cross_ring_pi_bonds}
benzene_options = {
    'include_nra': True,
    'nra_subspace': 'pi',
    'nra_fit_metric': 'frobenius',
    'max_alternatives': 24,
    'pi_min_occupation': 0.05,
    'conjugated_pi_max_path': 2,
}
mol_benzene, basis_benzene, scf_benzene, nbo_benzene, res_benzene = run_nbo('benzene', basis_label='sto-3g', constraints=benzene_constraints, options=benzene_options)
print_nra_summary(res_benzene)
print()
print_alternatives(res_benzene, limit=12)

NRA subspace       : pi
Fit metric         : frobenius
Spin fit           : none
Weight sum         : 1.000000000000
Residual           : 1.0038898791067454
Total residual     : 1.0038898791067454
Spin residual      : None
Prior              : {'active': False, 'mode': 'regularized', 'strength': 0.0, 'input_weights': None, 'normalized_weights': None, 'fixed': False, 'warnings': []}

rank   score_wt     nra_wt       score  e_count pi bonds             active
   1    37.629%    33.333%    41.04111     42.0 1-2, 3-4, 5-6        {('BD', 'pi', 2.0): 3}
   2    37.629%    33.333%    41.04111     42.0 1-6, 2-3, 4-5        {('BD', 'pi', 2.0): 3}
   3     8.129%    11.119%    40.65803     42.0 1-4, 2-3, 5-6        {('BD', 'pi', 2.0): 3}
   4     8.117%    11.107%    40.65765     42.0 1-6, 2-5, 3-4        {('BD', 'pi', 2.0): 3}
   5     8.117%    11.107%    40.65765     42.0 1-2, 3-6, 4-5        {('BD', 'pi', 2.0): 3}
   6     0.378%     0.000%    39.89111     42.0 1-4, 2-5, 3-6        {('BD', '

In [12]:
print(nbo_benzene.nra_report(level='summary', return_text=True))

Natural Resonance Analysis (NRA/NRT) Density-Fit Summary
NRA weights are density-fit weights; Score weights are Lewis ranking weights.

Subspace: pi  Metric: frobenius  Residual: 1.003890e+00  Relative residual: 1.693123e-01
Spin fit: none  Total residual: 1.003890e+00  Spin residual: n/a
Prior mode: regularized  Prior strength: 0  Prior active: False

Rank Signature            NRA weight  Score weight        Score     Residual   Sigma/Pi Pi bonds
--------------------------------------------------------------------------------------------------------
   1 pi:1-2,3-4,5-6         0.333333      0.376294     41.04111  2.02951e+00    18/3    1-2, 3-4, 5-6
   2 pi:1-6,2-3,4-5         0.333333      0.376294     41.04111  2.02951e+00    18/3    1-6, 2-3, 4-5
   3 pi:1-4,2-3,5-6         0.111195      0.081292     40.65803  2.33479e+00    18/3    1-4, 2-3, 5-6
   4 pi:1-6,2-5,3-4         0.111069      0.081169     40.65765  2.33511e+00    18/3    1-6, 2-5, 3-4
   5 pi:1-2,3-6,4-5         0.11106

In [13]:
prior_options = dict(benzene_options)
prior_options.update({
    'nra_prior_weights': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    'nra_prior_strength': 25.0,
})
_, _, _, nbo_benzene_prior, res_benzene_prior = run_nbo('benzene', basis_label='sto-3g', constraints=benzene_constraints, options=prior_options)
print_nra_summary(res_benzene_prior)
print()
print_alternatives(res_benzene_prior, limit=6)
print()
print(nbo_benzene_prior.nra_report(level='summary', return_text=True))

NRA subspace       : pi
Fit metric         : frobenius
Spin fit           : none
Weight sum         : 1.000000000000
Residual           : 1.7410198863004342
Total residual     : 1.7410198863004342
Spin residual      : None
Prior              : {'active': True, 'mode': 'regularized', 'strength': 25.0, 'input_weights': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'normalized_weights': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'fixed': False, 'warnings': []}

rank   score_wt     nra_wt       score  e_count pi bonds             active
   1    37.629%    89.247%    41.04111     42.0 1-6, 2-3, 4-5        {('BD', 'pi', 2.0): 3}
   2    37.629%     8.602%    41.04111     42.0 1-2, 3-4, 5-6        {('BD', 'pi', 2.0): 3}
   3     8.129%     0.001%    40.65803     42.0 1-4, 2-3, 5-6        {('BD', 'pi', 2.0): 3}
   4     8.117%     0.000%    40.65765     42.0 1-6, 2-5, 3-4        {('BD', 'pi', 2.0): 3}
   5     8.117%     0.000%    40.65765     42.0 1-2, 3-6, 4-5        {('BD', 'pi', 2.0): 3}
   6     0.378%     2.150%

## 7. Lone-Pair Donation and Charge-Separated Alternatives

Formate, nitrate, and ozone exercise the active-space model where pi bonds and donating lone pairs can be exchanged. These are useful examples for inspecting `active_nbo_list`, `active_pi_nbo_list`, and `active_lone_pair_nbo_list`.

In [14]:
lp_options = {
    'include_nra': True,
    'nra_subspace': 'pi',
    'nra_fit_metric': 'frobenius',
    'max_alternatives': 24,
    'pi_min_occupation': 0.05,
    'conjugated_pi_max_path': 2,
}
lp_cases = [
    ('formate', -1, 1, 'sto-3g', lp_options),
    ('nitrate', -1, 1, 'sto-3g', lp_options),
    ('ozone', 0, 1, 'sto-3g', dict(lp_options, lone_pair_min_occupation=1.0)),
]
lp_results = {}
for name, charge, mult, basis_label, options in lp_cases:
    lp_results[name] = run_nbo(name, charge=charge, multiplicity=mult, basis_label=basis_label, options=options)
    _, _, _, _, results = lp_results[name]
    donor_alts = [alt for alt in results['alternatives'] if alt['active_lone_pair_electron_pairs'] > 0]
    print(name)
    print('  primary counts        :', results['primary']['counts'])
    print('  alternatives          :', len(results['alternatives']))
    print('  LP-donation alternatives:', len(donor_alts))
    print('  NRA weight sum        :', f"{sum(results['nra']['weights']):.12f}")
    print()

formate
  primary counts        : {'CR': 3, 'LP': 6, 'BD': 3}
  alternatives          : 13
  LP-donation alternatives: 13
  NRA weight sum        : 1.000000000000

nitrate
  primary counts        : {'CR': 4, 'LP': 9, 'BD': 3}
  alternatives          : 24
  LP-donation alternatives: 24
  NRA weight sum        : 1.000000000000

ozone
  primary counts        : {'CR': 3, 'LP': 9}
  alternatives          : 19
  LP-donation alternatives: 19
  NRA weight sum        : 1.000000000000



In [15]:
_, _, _, _, res_formate = lp_results['formate']
print_alternatives(res_formate, limit=10)
print()
for alt in res_formate['alternatives'][:5]:
    print('rank', alt['rank'])
    print('  active pi bonds       :', alt['pi_bonds'])
    print('  active LP atoms       :', alt['active_lone_pair_atoms'])
    print('  formal charges        :', np.array(alt['formal_charges']))

rank   score_wt     nra_wt       score  e_count pi bonds             active
   1    17.667%    16.883%    23.38236     24.0 1-2                  {('LP', 'lone-pair', 2.0): 5, ('BD', 'pi', 2.0): 1}
   2    17.667%    13.759%    23.38236     24.0 1-3                  {('LP', 'lone-pair', 2.0): 5, ('BD', 'pi', 2.0): 1}
   3    17.667%     5.808%    23.38236     24.0 1-3                  {('LP', 'lone-pair', 2.0): 5, ('BD', 'pi', 2.0): 1}
   4    17.667%     8.933%    23.38236     24.0 1-2                  {('LP', 'lone-pair', 2.0): 5, ('BD', 'pi', 2.0): 1}
   5     3.744%     6.942%    22.99447     24.0 1-2                  {('LP', 'lone-pair', 2.0): 5, ('BD', 'pi', 2.0): 1}
   6     3.744%     3.818%    22.99447     24.0 1-3                  {('LP', 'lone-pair', 2.0): 5, ('BD', 'pi', 2.0): 1}
   7     3.744%    11.361%    22.99447     24.0 1-2                  {('LP', 'lone-pair', 2.0): 5, ('BD', 'pi', 2.0): 1}
   8     3.744%     8.236%    22.99447     24.0 1-3                  {('LP', 

## 8. NO Radical: Compact Open-Shell Diatomic Example

NO is a small open-shell example for spin diagnostics. It complements the allyl radical by showing radical behavior in a compact diatomic system.

In [16]:
no_options = {
    'include_nra': True,
    'nra_subspace': 'valence',
    'nra_fit_metric': 'frobenius',
    'max_alternatives': 12,
    'pi_min_occupation': 0.05,
}
mol_no, basis_no, scf_no, nbo_no, res_no = run_nbo('no', multiplicity=2, basis_label='sto-3g', options=no_options)
print_assignment_summary(res_no)
print()
print('Unpaired electrons:', res_no['unpaired_electrons'])
print('One-electron candidates:')
for c in res_no['nbo_candidates']:
    if c.get('electron_count', 2.0) == 1.0:
        print(c['index'], c['type'], c.get('subtype'), tuple(a+1 for a in c['atoms']), 'occ=', round(c['occupation'], 5), 'spin=', round(c.get('spin_occupation', 0.0), 5))
print()
print_nra_summary(res_no)

Primary counts        : {'CR': 2, 'LP': 3, 'BD': 3}
Candidate counts      : {'CR': 2, 'SOMO': 2, 'LP': 5, 'BD': 3, 'BD*': 2}
Selected electrons    : 15.0
Target electrons      : 15.0
Formal charges        : [ 0.5 -0.5]
Sigma/pi pairs        : 6 / 2

Unpaired electrons: 0.999999999999999
One-electron candidates:
2 SOMO one-electron (1,) occ= 1.34287 spin= 0.65713
3 LP radical-lone-pair (1,) occ= 1.34287 spin= 0.65713
6 SOMO one-electron (2,) occ= 1.65713 spin= 0.34287
7 LP radical-lone-pair (2,) occ= 1.65713 spin= 0.34287
14 BD pi (1, 2) occ= 1.0 spin= 1.0

NRA subspace       : valence
Fit metric         : frobenius
Spin fit           : total_spin
Weight sum         : 1.000000000000
Residual           : 1.1846854278544217
Total residual     : 0.8281662227846582
Spin residual      : 0.8471247077079065
Prior              : {'active': False, 'mode': 'regularized', 'strength': 0.0, 'input_weights': None, 'normalized_weights': None, 'fixed': False, 'warnings': []}


## 8. Summary: What to Inspect in Your Own Systems

For routine analysis, start with the foundation diagnostics and `npa_report()`, then inspect `nbo_report(level='full')`. For resonance questions, enable `include_nra=True` and compare `Score weight` against `NRA weight`. For open-shell systems, check `spin_fit`, `spin_residual_norm`, and the one-electron active-space fields. For donor-acceptor questions, use `donor_acceptor_diagnostics` as a density-coupling diagnostic rather than a second-order perturbation energy.